In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import udf

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df = spark.createDataFrame(data, columns)


In [0]:
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df = df.fillna({"amount": "1000"}).display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
print(type(df))

<class 'NoneType'>


In [0]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType

df = df.withColumn("amount", col("amount").cast(IntegerType())) \
       .withColumn("updated_date", to_date(col("updated_date")))
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType


new_data = [
    (3, "C003", "Tablet", "22000", "2024-01-06")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df_new = spark.createDataFrame(new_data, columns)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

df_new = df_new.withColumn("rn", row_number().over(window)) \
               .filter(col("rn") == 1) \
               .drop("rn")
df_new.display()

order_id,customer_id,product,amount,updated_date
3,C003,Tablet,22000,2024-01-06


In [0]:
df = df.withColumn("bonus", col("amount") * 0.1)

df = df.withColumn(
    "category",
    when(col("amount") >= 20000, "High").otherwise("Low")
)
df.display()

order_id,customer_id,product,amount,updated_date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,High
2,C002,Mobile,null,2024-01-02,null,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High
4,C004,Laptop,55000,2024-01-04,5500.0,High
5,C005,Headphones,null,2024-01-05,null,Low
6,C006,Camera,30000,2024-01-06,3000.0,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low
8,C008,Watch,8000,2024-01-07,800.0,Low


In [0]:
def amount_bucket(x):
    if x is None:
        return "Unknown"
    elif x < 10000:
        return "Low"
    elif 10000 <= x <= 30000:
        return "Medium"
    else:
        return "High"

bucket_udf = udf(amount_bucket)

df = df.withColumn("amount_bucket", bucket_udf(col("amount")))
df.display()

order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,null,2024-01-02,null,Low,Unknown
3,C003,Tablet,20000,2024-01-03,2000.0,High,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,null,2024-01-05,null,Low,Unknown
6,C006,Camera,30000,2024-01-06,3000.0,High,Medium
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("orders_table")

In [0]:
new_data = [
    (3, "C003", "Tablet", "22000", "2024-01-06")  # updated record
]

df_new = spark.createDataFrame(new_data, columns)
df_new.display()


order_id,customer_id,product,amount,updated_date
3,C003,Tablet,22000,2024-01-06


In [0]:
df_new = df_new.fillna({"amount": "1000"}) \
    .withColumn("amount", col("amount").cast(IntegerType())) \
    .withColumn("updated_date", to_date(col("updated_date")))


In [0]:
window = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

df_new = df_new.withColumn("rn", row_number().over(window)) \
               .filter(col("rn") == 1) \
               .drop("rn")



In [0]:
df_new.write.format("delta").mode("append").saveAsTable("orders_table")

In [0]:
dbutils.widgets.text("input_path", "/FileStore/orders_data/")
dbutils.widgets.text("last_loaded_date", "2024-01-05")

input_path = dbutils.widgets.get("input_path")
last_loaded_date = dbutils.widgets.get("last_loaded_date")

df_filtered = df.filter(col("updated_date") > last_loaded_date)

In [0]:
df.display()
df_new.display()
df_filtered.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


order_id,customer_id,product,amount,updated_date
3,C003,Tablet,22000,2024-01-06


order_id,customer_id,product,amount,updated_date
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07
